In [1]:
# 3D Particle Visualization with Rerun - Batch Processing
# Process all videos in cvpr_demo_figure_videos_processed and create separate visualizations

import os
import sys
import numpy as np
import jax
import jax.numpy as jnp
from jax.random import key as jkey

# JAX compilation cache setup
cache_dir = os.path.join(os.getcwd(), ".jax_cache")
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)
jax.config.update("jax_compilation_cache_dir", cache_dir)
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.experimental.compilation_cache.compilation_cache.set_cache_dir(cache_dir)

# Add workspace root to path to import from main script
workspace_root = os.path.dirname(os.getcwd())  # Go up from plotting_scripts to workspace root
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)

# Import rerun
import rerun as rr

# Import functions from the main tracking script
from dino_tracking_subsampling_dense_eval import (
    process_video,
    NUM_BLOBS,
    NUM_HYPERBLOBS_ORIGINAL,
    FOCAL_LENGTH,
    BLOB_COUNTING_THRESHOLD,
    RANDOM_SEED,
    DAVIS_3D_MOTION_PATH,
    DAVIS_SEGMASKS_PATH,
    DAVIS_RGB_PATH,
    DINO_PATH_TEMPLATE,
    EXPERIMENT_SAVE_DIR
)

# Override paths with correct locations for this system
import dino_tracking_subsampling_dense_eval as tracking_module

# Update paths to use cvpr_demo_figure_videos_processed directories
tracking_module.DAVIS_3D_MOTION_PATH = "/home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_npzs"
tracking_module.DINO_PATH_TEMPLATE = '/home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_dino/{}_dino_pca_per_pixel.npz'

# Set segmentation masks path
segmask_path = "/home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_segmasks"
if os.path.exists(segmask_path):
    tracking_module.DAVIS_SEGMASKS_PATH = segmask_path
    print(f"Using segmentation masks at: {segmask_path}")
else:
    print(f"Warning: Segmentation mask path not found: {segmask_path}")

# Enable SAM frame0 and set path
tracking_module.USE_SAM_FRAME0 = True
tracking_module.SAM_FRAME0_PATH_TEMPLATE = '/home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_SAM_frame0/{}_SAM_frame0.png'
print("Enabled SAM frame0 (using SAM segmentation masks)")

tracking_module.DAVIS_RGB_PATH = "/home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_rgb_frames"
print("Updated paths:")
print(f"  3D Motion: {tracking_module.DAVIS_3D_MOTION_PATH}")
print(f"  DINO Features: {tracking_module.DINO_PATH_TEMPLATE}")
print(f"  Segmentation Masks: {tracking_module.DAVIS_SEGMASKS_PATH}")
print(f"  RGB Frames: {tracking_module.DAVIS_RGB_PATH}")
print(f"  SAM Frame0: {tracking_module.SAM_FRAME0_PATH_TEMPLATE}")

print("\nSetup complete!")


Using segmentation masks at: /home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_segmasks
Enabled SAM frame0 (using SAM segmentation masks)
Updated paths:
  3D Motion: /home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_npzs
  DINO Features: /home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_dino/{}_dino_pca_per_pixel.npz
  Segmentation Masks: /home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_segmasks
  RGB Frames: /home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_rgb_frames
  SAM Frame0: /home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_SAM_frame0/{}_SAM_frame0.png

Setup complete!


In [2]:
# Process all videos and create separate Rerun visualizations for each
import glob
import cv2
from scipy.linalg import eigh
import matplotlib.cm as cm

# Get all video names from the npz files
npz_dir = "/home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_npzs"
npz_files = glob.glob(os.path.join(npz_dir, "*_3d_data.npz"))
# video_names = [os.path.basename(f).replace("_3d_data.npz", "") for f in npz_files]
# video_names.sort()

video_names = [
    "belt",
    "cloth_bag",
    "gray_jacket",
    "jello_trim",
]


print(f"Found {len(video_names)} videos to process:")
for vn in video_names:
    print(f"  - {vn}")

# Configuration
SUBSAMPLING_PERCENTAGE = 12.5  # Use 12.5% subsampling for faster processing
VISUALIZATION_PARAMS = {
    "BACKGROUND_DARKENING_FACTOR": 1.0,
    "POINT_SUBSAMPLE_FACTOR": 1,
    "POINT_RADIUS": 0.015,
    "ELLIPSOID_SCALE": 2.0,
    "MAX_PARTICLES_TO_SHOW": 500,
    "SIZE_FILTER_THRESHOLD": 5.0,
    "PRINCIPAL_AXIS_RATIO_THRESHOLD": 4.0,
    "MIN_WEIGHT_PERCENTILE": 30.0,  # Keep particles above this percentile of weights
}

# Helper function to create ellipsoid mesh
def create_ellipsoid_mesh(mean, cov, scale=1.0, num_points=32):
    """Create ellipsoid mesh vertices from mean and covariance matrix"""
    eigenvals, eigenvecs = eigh(cov)
    eigenvals = np.maximum(eigenvals, 1e-6)
    radii = np.sqrt(eigenvals) * scale
    
    u = np.linspace(0, 2 * np.pi, num_points)
    v = np.linspace(0, np.pi, num_points)
    u, v = np.meshgrid(u, v)
    
    x = np.sin(v) * np.cos(u)
    y = np.sin(v) * np.sin(u)
    z = np.cos(v)
    
    sphere_points = np.stack([x.flatten(), y.flatten(), z.flatten()], axis=1)
    sphere_points = sphere_points * radii
    sphere_points = sphere_points @ eigenvecs.T
    ellipsoid_points = sphere_points + mean
    
    n_u, n_v = num_points, num_points
    faces = []
    for i in range(n_v - 1):
        for j in range(n_u - 1):
            idx = i * n_u + j
            faces.append([idx, idx + 1, idx + n_u])
            faces.append([idx + 1, idx + n_u + 1, idx + n_u])
    
    return ellipsoid_points, np.array(faces)

print(f"\n{'='*80}")
print(f"Starting batch processing of {len(video_names)} videos")
print(f"{'='*80}\n")


Found 4 videos to process:
  - belt
  - cloth_bag
  - gray_jacket
  - jello_trim

Starting batch processing of 4 videos



In [3]:
# Main loop: Process each video and create visualization
all_results = {}

for video_idx, VIDEO_NAME in enumerate(video_names):
    print(f"\n{'='*80}")
    print(f"Processing video {video_idx + 1}/{len(video_names)}: {VIDEO_NAME}")
    print(f"{'='*80}")
    
    try:
        # Process the video
        result = process_video(
            video_name=VIDEO_NAME,
            subsampling_percentage=SUBSAMPLING_PERCENTAGE,
            subsampled_indices=None,
            run_save_dir=None
        )
        
        if result is None:
            print(f"⚠️  Failed to process {VIDEO_NAME}, skipping...")
            continue
        
        all_results[VIDEO_NAME] = result
        tracking_data = result['tracking_data']
        segmentation_masks = result['segmentation_masks']
        img_dims = result['img_dims']
        
        print(f"✓ Processed {VIDEO_NAME}: {len(tracking_data)} frames, dimensions {img_dims}")
        
        # Extract particle data
        particle_means = []
        particle_covs = []
        particle_weights = []
        is_object_particle = []
        
        fx = fy = FOCAL_LENGTH
        cx = img_dims[1] / 2.0
        cy = img_dims[0] / 2.0
        
        # Determine object particles from frame 0
        frame0 = tracking_data[0]
        blob_assignments_frame0 = frame0['blob_assignments']
        n_blobs_frame0 = frame0['n_blobs']
        gt_mask_frame0 = segmentation_masks[0]
        
        blob_pixel_counts_frame0 = np.bincount(
            blob_assignments_frame0[blob_assignments_frame0 < n_blobs_frame0],
            minlength=n_blobs_frame0
        )
        significant_blobs = np.where(blob_pixel_counts_frame0 >= BLOB_COUNTING_THRESHOLD)[0]
        
        blob_means_frame0 = frame0['blob_means']
        x_2d = (blob_means_frame0[:, 0] / (blob_means_frame0[:, 2] + 1e-8)) * fx + cx
        y_2d = (blob_means_frame0[:, 1] / (blob_means_frame0[:, 2] + 1e-8)) * fy + cy
        x_2d = np.clip(x_2d.astype(int), 0, img_dims[1] - 1)
        y_2d = np.clip(y_2d.astype(int), 0, img_dims[0] - 1)
        
        object_blobs_frame0 = []
        for blob_idx in significant_blobs:
            pixel_idx = y_2d[blob_idx] * img_dims[1] + x_2d[blob_idx]
            is_on_mask = pixel_idx < len(gt_mask_frame0) and gt_mask_frame0[pixel_idx]
            if is_on_mask:
                object_blobs_frame0.append(blob_idx)
        
        object_blobs_frame0 = np.array(object_blobs_frame0)
        
        # Extract data for all frames
        for frame_idx in range(len(tracking_data)):
            frame = tracking_data[frame_idx]
            blob_means = np.array(frame['blob_means'])
            blob_covs = np.array(frame['blob_covs'])
            blob_weights = np.array(frame['blob_weights'])
            n_blobs = frame['n_blobs']
            
            is_object = np.zeros(n_blobs, dtype=bool)
            for obj_idx in object_blobs_frame0:
                if obj_idx < n_blobs:
                    is_object[obj_idx] = True
            
            particle_means.append(blob_means)
            particle_covs.append(blob_covs)
            particle_weights.append(blob_weights)
            is_object_particle.append(is_object)
        
        print(f"  Extracted {len(particle_means)} frames of particle data")
        
        # Load RGB frames
        rgb_base_path = "/home/esli/GenParticles_neural_stimulus/assets/cvpr_demo_figure_videos_processed/cvpr_demo_figure_rgb_frames"
        rgb_dir = os.path.join(rgb_base_path, VIDEO_NAME)
        rgb_files = sorted(glob.glob(os.path.join(rgb_dir, "*.jpg")))
        if len(rgb_files) == 0:
            rgb_files = sorted(glob.glob(os.path.join(rgb_dir, "*.png")))
        
        frame_indices_to_visualize = list(range(0, len(tracking_data), 1))
        rgb_frames = []
        if len(rgb_files) > 0:
            for frame_idx in frame_indices_to_visualize:
                if frame_idx < len(rgb_files):
                    frame = cv2.imread(rgb_files[frame_idx])
                    if frame is not None:
                        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        rgb_frames.append(frame)
        
        # Compute average RGB colors per particle
        particle_rgb_colors = []
        for vis_idx, frame_idx in enumerate(frame_indices_to_visualize):
            frame_data = tracking_data[frame_idx]
            blob_assignments = frame_data['blob_assignments']
            n_blobs = frame_data['n_blobs']
            blob_assignments_2d = blob_assignments.reshape(img_dims)
            
            if vis_idx < len(rgb_frames):
                rgb_frame = rgb_frames[vis_idx]
                if rgb_frame.shape[:2] != img_dims:
                    rgb_frame = cv2.resize(rgb_frame, (img_dims[1], img_dims[0]), interpolation=cv2.INTER_LINEAR)
            else:
                rgb_frame = np.full((img_dims[0], img_dims[1], 3), 128, dtype=np.uint8)
            
            blob_colors = np.zeros((n_blobs, 3), dtype=np.float32)
            for blob_idx in range(n_blobs):
                mask = (blob_assignments_2d == blob_idx)
                if np.any(mask):
                    blob_colors[blob_idx] = np.mean(rgb_frame[mask], axis=0)
                else:
                    blob_colors[blob_idx] = [128, 128, 128]
            
            particle_rgb_colors.append(blob_colors.astype(np.uint8))
        
        print(f"  Loaded {len(rgb_frames)} RGB frames")
        
        # Create Rerun visualization for this video
        rerun_session_name = f"particle_visualization_{VIDEO_NAME}"
        rr.init(rerun_session_name, spawn=True)
        rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_UP)
        
        # Set up Blueprint with two side-by-side 3D views
        try:
            import rerun.blueprint as rrb
            blueprint = rrb.Blueprint(
                rrb.Horizontal(
                    rrb.Spatial3DView(
                        origin="/world",
                        name="3D View 1",
                        background=[255, 255, 255],
                    ),
                    rrb.Spatial3DView(
                        origin="/world",
                        name="3D View 2",
                        background=[255, 255, 255],
                    ),
                ),
                collapse_panels=True,
            )
            rr.send_blueprint(blueprint)
        except Exception as e:
            print(f"  Warning: Could not set up side-by-side views: {e}")
        
        print(f"  Creating visualization for {len(frame_indices_to_visualize)} frames...")
        
        # Visualize each frame
        for vis_idx, frame_idx in enumerate(frame_indices_to_visualize):
            frame_data = tracking_data[frame_idx]
            rr.set_time_sequence("frame", vis_idx)
            
            # Log RGB frame
            if vis_idx < len(rgb_frames):
                rgb_frame = rgb_frames[vis_idx]
                if rgb_frame.shape[:2] != img_dims:
                    rgb_frame = cv2.resize(rgb_frame, (img_dims[1], img_dims[0]), interpolation=cv2.INTER_LINEAR)
                if rgb_frame.dtype != np.uint8:
                    rgb_frame = rgb_frame.astype(np.uint8)
                rr.log("world/rgb_frame", rr.Image(rgb_frame))
            
            # Log point cloud
            datapoint_positions = frame_data['datapoint_positions']
            blob_assignments = frame_data['blob_assignments']
            n_blobs = frame_data['n_blobs']
            
            valid_mask = blob_assignments < n_blobs
            valid_positions = datapoint_positions[valid_mask]
            valid_assignments = blob_assignments[valid_mask]
            
            if VISUALIZATION_PARAMS["POINT_SUBSAMPLE_FACTOR"] > 1:
                subsample_indices = np.arange(0, len(valid_positions), VISUALIZATION_PARAMS["POINT_SUBSAMPLE_FACTOR"])
                valid_positions = valid_positions[subsample_indices]
                valid_assignments = valid_assignments[subsample_indices]
            
            valid_positions_flipped = valid_positions.copy()
            valid_positions_flipped[:, 0] = -valid_positions_flipped[:, 0]
            valid_positions_flipped[:, 1] = -valid_positions_flipped[:, 1]
            
            is_object = is_object_particle[frame_idx]
            is_object_pixel = is_object[valid_assignments]
            
            # Get RGB colors for point cloud
            if vis_idx < len(rgb_frames):
                rgb_frame_for_colors = rgb_frames[vis_idx]
                if rgb_frame_for_colors.shape[:2] != img_dims:
                    rgb_frame_for_colors = cv2.resize(rgb_frame_for_colors, (img_dims[1], img_dims[0]), interpolation=cv2.INTER_LINEAR)
                
                fx = fy = FOCAL_LENGTH
                cx = img_dims[1] / 2.0
                cy = img_dims[0] / 2.0
                
                x_2d = (valid_positions[:, 0] / (valid_positions[:, 2] + 1e-8)) * fx + cx
                y_2d = (valid_positions[:, 1] / (valid_positions[:, 2] + 1e-8)) * fy + cy
                x_2d = np.clip(x_2d.astype(int), 0, img_dims[1] - 1)
                y_2d = np.clip(y_2d.astype(int), 0, img_dims[0] - 1)
                
                original_colors = rgb_frame_for_colors[y_2d, x_2d]
            else:
                original_colors = np.full((len(valid_positions), 3), 128, dtype=np.uint8)
            
            original_colors_rgba = np.zeros((len(valid_positions), 4), dtype=np.uint8)
            original_colors_rgba[:, :3] = original_colors
            original_colors_rgba[:, 3] = 255
            
            rr.log(
                "world/point_clouds/original_rgb",
                rr.Points3D(
                    positions=valid_positions_flipped,
                    colors=original_colors_rgba,
                    radii=VISUALIZATION_PARAMS["POINT_RADIUS"]
                )
            )
            
            # Log particles as ellipsoids
            means = particle_means[frame_idx].copy()
            covs = particle_covs[frame_idx].copy()
            weights = particle_weights[frame_idx].copy()
            is_object_particles = is_object_particle[frame_idx].copy()
            n_particles_original = len(means)
            n_particles = n_particles_original
            
            # Apply weight-based culling to reduce flicker from low-weight particles
            weight_mask = None
            if VISUALIZATION_PARAMS["MIN_WEIGHT_PERCENTILE"] > 0 and n_particles > 0:
                weight_threshold = np.percentile(weights, VISUALIZATION_PARAMS["MIN_WEIGHT_PERCENTILE"])
                weight_mask = weights >= weight_threshold
                n_filtered = np.sum(~weight_mask)
                if n_filtered > 0:
                    means = means[weight_mask]
                    covs = covs[weight_mask]
                    weights = weights[weight_mask]
                    is_object_particles = is_object_particles[weight_mask]
                    n_particles = len(means)
                    if vis_idx == 0 or vis_idx % 10 == 0:  # Print every 10th frame to avoid spam
                        print(f"Frame {frame_idx}: Weight-based culling filtered out {n_filtered} particles (kept {n_particles} above {VISUALIZATION_PARAMS['MIN_WEIGHT_PERCENTILE']}th percentile, threshold={weight_threshold:.6f})")
            
            # Apply coordinate flip
            means[:, 0] = -means[:, 0]
            means[:, 1] = -means[:, 1]
            flip_matrix = np.array([[-1, 0, 0], [0, -1, 0], [0, 0, 1]], dtype=np.float32)
            for i in range(len(covs)):
                covs[i] = flip_matrix @ covs[i] @ flip_matrix.T
            
            # Get colors for particles (RGB colors) - get full array first
            if vis_idx < len(particle_rgb_colors):
                rgb_colors_full = particle_rgb_colors[vis_idx].copy()
            else:
                rgb_colors_full = np.full((n_particles_original, 3), 128, dtype=np.uint8)
            
            # Get cluster colors for particles - get full array first
            hyperblob_assignments_frame = frame_data['hyperblob_assignments']
            n_hyperblobs = frame_data['n_hyperblobs']
            cluster_colormap = cm.get_cmap('tab20')
            cluster_colors_rgb = np.zeros((n_hyperblobs + 1, 3), dtype=np.uint8)
            for cluster_idx in range(n_hyperblobs + 1):
                color_rgba = cluster_colormap(cluster_idx % 20)
                cluster_colors_rgb[cluster_idx] = (np.array(color_rgba[:3]) * 255).astype(np.uint8)
            
            # Map each particle to its cluster color (full array)
            cluster_colors_full = cluster_colors_rgb[hyperblob_assignments_frame]  # (n_blobs, 3)
            
            # Apply weight mask to colors if weight filtering was applied
            if weight_mask is not None:
                rgb_colors = rgb_colors_full[weight_mask]
                cluster_colors = cluster_colors_full[weight_mask]
            else:
                rgb_colors = rgb_colors_full
                cluster_colors = cluster_colors_full
            
            # Limit particles if needed
            if n_particles > VISUALIZATION_PARAMS["MAX_PARTICLES_TO_SHOW"]:
                indices = np.argsort(weights)[-VISUALIZATION_PARAMS["MAX_PARTICLES_TO_SHOW"]:]
                means = means[indices].copy()
                covs = covs[indices].copy()
                weights = weights[indices].copy()
                rgb_colors = rgb_colors[indices].copy()
                cluster_colors = cluster_colors[indices].copy()
                is_object_particles = is_object_particles[indices].copy()
                n_particles = VISUALIZATION_PARAMS["MAX_PARTICLES_TO_SHOW"]
            
            # Filter particles by size and shape
            volumes = np.array([np.sqrt(np.linalg.det(cov)) for cov in covs])
            reference_volume = np.percentile(volumes, 75)
            
            valid_particle_mask = np.ones(n_particles, dtype=bool)
            
            if VISUALIZATION_PARAMS["SIZE_FILTER_THRESHOLD"] > 0:
                size_filtered = volumes > (VISUALIZATION_PARAMS["SIZE_FILTER_THRESHOLD"] * reference_volume)
                valid_particle_mask = valid_particle_mask & ~size_filtered
            
            if VISUALIZATION_PARAMS["PRINCIPAL_AXIS_RATIO_THRESHOLD"] > 0:
                for i in range(n_particles):
                    cov = covs[i]
                    eigenvals, eigenvecs = eigh(cov)
                    eigenvals = np.maximum(eigenvals, 1e-10)
                    eigenvals_sorted = np.sort(eigenvals)[::-1]
                    first_principal_axis = np.sqrt(eigenvals_sorted[0])
                    second_principal_axis = np.sqrt(eigenvals_sorted[1])
                    if second_principal_axis > 1e-6:
                        axis_ratio = first_principal_axis / second_principal_axis
                        if axis_ratio >= VISUALIZATION_PARAMS["PRINCIPAL_AXIS_RATIO_THRESHOLD"]:
                            valid_particle_mask[i] = False
            
            means = means[valid_particle_mask]
            covs = covs[valid_particle_mask]
            weights = weights[valid_particle_mask]
            rgb_colors = rgb_colors[valid_particle_mask]
            cluster_colors = cluster_colors[valid_particle_mask]
            is_object_particles = is_object_particles[valid_particle_mask]
            n_particles = len(means)
            
            # Compute opacity
            volumes = np.array([np.sqrt(np.linalg.det(cov)) for cov in covs])
            if volumes.max() > volumes.min():
                normalized_volumes = (volumes - volumes.min()) / (volumes.max() - volumes.min())
            else:
                normalized_volumes = np.ones(n_particles)
            opacity_from_volume = 1.0 - normalized_volumes * 0.75
            opacity_from_volume = np.clip(opacity_from_volume, 0.25, 1.0)
            opacity_values = opacity_from_volume
            
            # Log RGB-colored particles
            for i in range(n_particles):
                mean = means[i]
                cov = covs[i]
                color_rgb = rgb_colors[i]
                opacity = opacity_values[i]
                
                vertices, faces = create_ellipsoid_mesh(mean, cov, scale=VISUALIZATION_PARAMS["ELLIPSOID_SCALE"])
                num_vertices = len(vertices)
                
                alpha_uint8 = int(opacity * 255)
                vertex_colors = np.zeros((num_vertices, 4), dtype=np.uint8)
                vertex_colors[:, :3] = color_rgb
                vertex_colors[:, 3] = alpha_uint8
                
                rr.log(
                    f"world/rgb_particles/ellipsoid_{i}",
                    rr.Mesh3D(
                        vertex_positions=vertices,
                        triangle_indices=faces,
                        vertex_colors=vertex_colors
                    )
                )
            
            # Log cluster-colored particles
            for i in range(n_particles):
                mean = means[i]
                cov = covs[i]
                color_rgb = cluster_colors[i]
                opacity = opacity_values[i]
                
                vertices, faces = create_ellipsoid_mesh(mean, cov, scale=VISUALIZATION_PARAMS["ELLIPSOID_SCALE"])
                num_vertices = len(vertices)
                
                alpha_uint8 = int(opacity * 255)
                vertex_colors = np.zeros((num_vertices, 4), dtype=np.uint8)
                vertex_colors[:, :3] = color_rgb
                vertex_colors[:, 3] = alpha_uint8
                
                rr.log(
                    f"world/cluster_particles/ellipsoid_{i}",
                    rr.Mesh3D(
                        vertex_positions=vertices,
                        triangle_indices=faces,
                        vertex_colors=vertex_colors
                    )
                )
        
        print(f"✓ Created Rerun visualization: {rerun_session_name}")
        print(f"  Open Rerun viewer to see visualization for {VIDEO_NAME}")
        
    except Exception as e:
        print(f"⚠️  Error processing {VIDEO_NAME}: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*80}")
print(f"Batch processing complete!")
print(f"Successfully processed {len(all_results)}/{len(video_names)} videos")
print(f"{'='*80}")
print(f"\nEach video has its own Rerun visualization window:")
for vn in all_results.keys():
    print(f"  - {vn}: particle_visualization_{vn}")



Processing video 1/4: belt

Processing: belt | Subsampling: 12.5%
Running initial Gibbs sweeps...
Running tracking...
JIT compiling tracking function...
Measuring FPS after JIT compilation...
Tracking FPS: 3.63 frames/second (total time: 6.61s for 24 frames)


Dense Evaluating Blob Assignments: 100%|██████████| 25/25 [00:05<00:00,  4.56it/s]


Computing error rates...
is_mask_subsampled? ~~~~~~~~~~~> : False

📊 Processing dataset: belt

📈 Trial Results for belt:
Trial    Recall (%)   Precision  FPR      Jaccard  Accuracy   MW-F1-A    MW-F1-F    MW-J-A   MW-J-F   MW-Acc-A   MW-Acc-F   AUC     
--------------------------------------------------------------------------------------------------------------------------------------
1        55.57        0.948      0.003    0.540    0.952      0.901      0.811      0.822    0.685    0.941      0.935      0.005   
--------------------------------------------------------------------------------------------------------------------------------------
Note: MW-F1-A = Matter-Weighted F1 (Adaptive weights), MW-F1-F = Matter-Weighted F1 (Fixed frame-0 weights)
      MW-J-A = Matter-Weighted Jaccard (Adaptive weights), MW-J-F = Matter-Weighted Jaccard (Fixed frame-0 weights)
      MW-Acc-A = Matter-Weighted Accuracy (Adaptive weights), MW-Acc-F = Matter-Weighted Accuracy (Fixed frame-0 weight

/var/tmp/ipykernel_195188/2356603388.py:162: DeprecationWarning: Use `set_time(sequence=…)` instead.
    See: https://www.rerun.io/docs/reference/migration/migration-0-23 for more details.
  rr.set_time_sequence("frame", vis_idx)
/var/tmp/ipykernel_195188/2356603388.py:265: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cluster_colormap = cm.get_cmap('tab20')


Frame 10: Weight-based culling filtered out 150 particles (kept 350 above 30.0th percentile, threshold=0.000006)
Frame 20: Weight-based culling filtered out 150 particles (kept 350 above 30.0th percentile, threshold=0.000006)
✓ Created Rerun visualization: particle_visualization_belt
  Open Rerun viewer to see visualization for belt

Processing video 2/4: cloth_bag

Processing: cloth_bag | Subsampling: 12.5%
Running initial Gibbs sweeps...
Running tracking...
JIT compiling tracking function...
Measuring FPS after JIT compilation...
Tracking FPS: 6.32 frames/second (total time: 8.87s for 56 frames)


...

Frame 10: Weight-based culling filtered out 150 particles (kept 348 above 30.0th percentile, threshold=0.001080)
Frame 20: Weight-based culling filtered out 150 particles (kept 348 above 30.0th percentile, threshold=0.000814)
Frame 30: Weight-based culling filtered out 150 particles (kept 348 above 30.0th percentile, threshold=0.000600)
✓ Created Rerun visualization: particle_visualization_gray_jacket
  Open Rerun viewer to see visualization for gray_jacket

Processing video 4/4: jello_trim

Processing: jello_trim | Subsampling: 12.5%
Limited to first 50 frames for jello_trim
Running initial Gibbs sweeps...
Running tracking...
JIT compiling tracking function...
Measuring FPS after JIT compilation...
Tracking FPS: 6.03 frames/second (total time: 8.13s for 49 frames)


Dense Evaluating Blob Assignments: 100%|██████████| 50/50 [00:10<00:00,  4.94it/s]


Computing error rates...
is_mask_subsampled? ~~~~~~~~~~~> : False

📊 Processing dataset: jello_trim

📈 Trial Results for jello_trim:
Trial    Recall (%)   Precision  FPR      Jaccard  Accuracy   MW-F1-A    MW-F1-F    MW-J-A   MW-J-F   MW-Acc-A   MW-Acc-F   AUC     
--------------------------------------------------------------------------------------------------------------------------------------
1        78.10        0.954      0.005    0.753    0.968      0.971      0.974      0.943    0.949    0.985      0.987      0.008   
--------------------------------------------------------------------------------------------------------------------------------------
Note: MW-F1-A = Matter-Weighted F1 (Adaptive weights), MW-F1-F = Matter-Weighted F1 (Fixed frame-0 weights)
      MW-J-A = Matter-Weighted Jaccard (Adaptive weights), MW-J-F = Matter-Weighted Jaccard (Fixed frame-0 weights)
      MW-Acc-A = Matter-Weighted Accuracy (Adaptive weights), MW-Acc-F = Matter-Weighted Accuracy (Fixed fr

/var/tmp/ipykernel_195188/2356603388.py:162: DeprecationWarning: Use `set_time(sequence=…)` instead.
    See: https://www.rerun.io/docs/reference/migration/migration-0-23 for more details.
  rr.set_time_sequence("frame", vis_idx)
/var/tmp/ipykernel_195188/2356603388.py:265: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cluster_colormap = cm.get_cmap('tab20')


Frame 10: Weight-based culling filtered out 150 particles (kept 348 above 30.0th percentile, threshold=0.001315)
Frame 20: Weight-based culling filtered out 150 particles (kept 348 above 30.0th percentile, threshold=0.001336)
Frame 30: Weight-based culling filtered out 150 particles (kept 348 above 30.0th percentile, threshold=0.001308)
Frame 40: Weight-based culling filtered out 150 particles (kept 348 above 30.0th percentile, threshold=0.001341)
✓ Created Rerun visualization: particle_visualization_jello_trim
  Open Rerun viewer to see visualization for jello_trim

Batch processing complete!
Successfully processed 4/4 videos

Each video has its own Rerun visualization window:
  - belt: particle_visualization_belt
  - cloth_bag: particle_visualization_cloth_bag
  - gray_jacket: particle_visualization_gray_jacket
  - jello_trim: particle_visualization_jello_trim
